# 04 — 예측 성능 비교 분석
ARIMA / XGBoost / LSTM 세 모델의 예측 결과를 다각도로 비교합니다.

| 분석 항목 | 셀 |
|---|---|
| SMAPE 계산 및 비교표 | 3 |
| 카테고리별 모델 성능 순위 | 4 |
| SMAPE 히트맵 | 5 |
| 예측값 vs 실제값 시각화 | 6 |
| 오차 분포 (Boxplot / Violin) | 7 |
| 날짜별 오차 추이 | 8 |
| 종합 인사이트 | 9 |

## 1. 라이브러리

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# 시각화 스타일
plt.rcParams.update({
    'figure.dpi'     : 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid'      : True,
    'grid.alpha'     : 0.3,
    'font.size'      : 9,
})

MODEL_COLORS = {
    'ARIMA'  : '#378ADD',
    'XGBoost': '#1D9E75',
    'LSTM'   : '#D85A30',
}
MODEL_KEYS = ['ARIMA', 'XGBoost', 'LSTM']

print("라이브러리 로드 완료")

## 2. 데이터 로드

`modeling.ipynb`에서 저장한 `predictions.csv`를 불러옵니다.

In [ ]:
pred_df = pd.read_csv('../data/predictions.csv', parse_dates=['date'])

# 컬럼명 통일 (modeling.ipynb 저장 형식)
# date | category | actual | pred_arima | pred_xgb | pred_lstm
pred_df.columns = pred_df.columns.str.strip()

CATEGORIES = pred_df['category'].unique().tolist()
TOP_N      = len(CATEGORIES)

print("pred_df shape:", pred_df.shape)
print("카테고리:", CATEGORIES)
print()
pred_df.head()

## 3. SMAPE 계산

수요가 0인 날이 있으므로 MAPE 대신 SMAPE를 사용합니다.  
실제·예측 모두 0인 경우 오차 = 0으로 처리합니다.

In [ ]:
def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom  = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask   = denom != 0
    error  = np.where(mask, np.abs(y_true - y_pred) / denom * 100, 0.0)
    return float(np.mean(error))


def smape_pointwise(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """날짜별 SMAPE (분포 분석용)"""
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom  = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask   = denom != 0
    return np.where(mask, np.abs(y_true - y_pred) / denom * 100, 0.0)


# ── 카테고리 × 모델 SMAPE 계산 ───────────────────────
col_map = {'ARIMA': 'pred_arima', 'XGBoost': 'pred_xgb', 'LSTM': 'pred_lstm'}

rows = []
for cat in CATEGORIES:
    sub = pred_df[pred_df['category'] == cat].dropna(subset=list(col_map.values()))
    row = {'category': cat}
    for model, col in col_map.items():
        row[f'smape_{model}'] = smape(sub['actual'], sub[col])
    row['best_model'] = min(MODEL_KEYS, key=lambda m: row[f'smape_{m}'])
    rows.append(row)

smape_df = pd.DataFrame(rows)
smape_df['smape_mean'] = smape_df[[f'smape_{m}' for m in MODEL_KEYS]].mean(axis=1)

# ── 전체 평균 행 추가 ────────────────────────────────
avg_row = {'category': '[ 전체 평균 ]'}
for m in MODEL_KEYS:
    avg_row[f'smape_{m}'] = smape_df[f'smape_{m}'].mean()
avg_row['best_model']  = min(MODEL_KEYS, key=lambda m: avg_row[f'smape_{m}'])
avg_row['smape_mean']  = smape_df['smape_mean'].mean()
smape_display = pd.concat([smape_df, pd.DataFrame([avg_row])], ignore_index=True)

print("SMAPE 비교표 (단위: %, 낮을수록 좋음)")
print("=" * 75)
print(smape_display[
    ['category', 'smape_ARIMA', 'smape_XGBoost', 'smape_LSTM', 'best_model']
].to_string(index=False, float_format=lambda x: f'{x:.2f}'))

## 4. 카테고리별 모델 성능 순위

In [ ]:
# ── 카테고리별 1·2·3위 ──────────────────────────────
rank_rows = []
for _, row in smape_df.iterrows():
    scores = {m: row[f'smape_{m}'] for m in MODEL_KEYS}
    ranked = sorted(scores, key=scores.get)
    rank_rows.append({
        'category': row['category'],
        '1위': f"{ranked[0]} ({scores[ranked[0]]:.1f}%)",
        '2위': f"{ranked[1]} ({scores[ranked[1]]:.1f}%)",
        '3위': f"{ranked[2]} ({scores[ranked[2]]:.1f}%)",
        '1위↔3위 격차': round(scores[ranked[2]] - scores[ranked[0]], 2),
    })
rank_df = pd.DataFrame(rank_rows)

print("카테고리별 모델 성능 순위")
print(rank_df.to_string(index=False))
print()

# ── 모델별 1위 획득 횟수 ─────────────────────────────
win_counts = smape_df['best_model'].value_counts()
print("── 모델별 1위 획득 카테고리 수 ──")
for m in MODEL_KEYS:
    cnt = win_counts.get(m, 0)
    bar = '█' * cnt
    print(f"  {m:<10}: {bar} ({cnt}개)")

# ── 시각화 ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 좌: 카테고리별 모델 SMAPE 그룹 바
ax = axes[0]
x = np.arange(len(smape_df))
w = 0.25
for i, m in enumerate(MODEL_KEYS):
    ax.bar(x + i*w, smape_df[f'smape_{m}'], width=w,
           label=m, color=MODEL_COLORS[m], alpha=0.85)
ax.set_xticks(x + w)
ax.set_xticklabels(
    [c[:18] for c in smape_df['category']],
    rotation=45, ha='right', fontsize=7
)
ax.set_ylabel('SMAPE (%)')
ax.set_title('카테고리별 모델 SMAPE')
ax.legend()

# 우: 모델별 1위 파이 차트
ax = axes[1]
labels = [m for m in MODEL_KEYS if m in win_counts]
sizes  = [win_counts[m] for m in labels]
colors = [MODEL_COLORS[m] for m in labels]
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.0f%%',
       startangle=90, textprops={'fontsize': 10})
ax.set_title('모델별 1위 비율')

plt.tight_layout()
plt.show()

## 5. SMAPE 히트맵

카테고리 × 모델을 한눈에 비교합니다.  
초록(낮은 오차) → 빨강(높은 오차) 순입니다.

In [ ]:
heat = smape_df.set_index('category')[[f'smape_{m}' for m in MODEL_KEYS]].copy()
heat.columns = MODEL_KEYS

fig, ax = plt.subplots(figsize=(7, TOP_N * 0.55 + 1.5))

im = ax.imshow(heat.values, aspect='auto', cmap='RdYlGn_r',
               vmin=heat.values.min(), vmax=heat.values.max())
plt.colorbar(im, ax=ax, label='SMAPE (%)', shrink=0.8)

ax.set_xticks(range(len(MODEL_KEYS)))
ax.set_xticklabels(MODEL_KEYS, fontsize=10)
ax.set_yticks(range(len(heat)))
ax.set_yticklabels(heat.index, fontsize=8)

# 셀 안에 수치 표기 + 최솟값 강조
for r_i, cat in enumerate(heat.index):
    row_vals = heat.loc[cat].values
    best_c   = int(np.argmin(row_vals))
    for c_i, val in enumerate(row_vals):
        weight = 'bold' if c_i == best_c else 'normal'
        marker = ' ★' if c_i == best_c else ''
        ax.text(c_i, r_i, f'{val:.1f}{marker}',
                ha='center', va='center', fontsize=8,
                fontweight=weight,
                color='white' if val > heat.values.mean() else 'black')

ax.set_title('SMAPE Heatmap  (★ = 카테고리 최저 오차 모델)', pad=12)
plt.tight_layout()
plt.show()

## 6. 예측값 vs 실제값 시각화

카테고리별로 실제 수요와 세 모델 예측값을 시계열로 비교합니다.

In [ ]:
n_cols = 2
n_rows = (TOP_N + 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
axes = axes.flatten()

for i, cat in enumerate(CATEGORIES):
    ax  = axes[i]
    sub = pred_df[pred_df['category'] == cat].sort_values('date')

    ax.plot(sub['date'], sub['actual'],
            color='black', linewidth=1.3, label='실제값', zorder=5)

    for m, col in col_map.items():
        s = smape_df.loc[smape_df['category'] == cat, f'smape_{m}'].values[0]
        ax.plot(sub['date'], sub[col],
                color=MODEL_COLORS[m], linewidth=0.9,
                linestyle='--', alpha=0.85,
                label=f'{m} ({s:.1f}%)')

    ax.set_title(cat, fontsize=9)
    ax.set_ylabel('Demand', fontsize=8)
    ax.tick_params(axis='x', rotation=40, labelsize=7)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.legend(fontsize=7, loc='upper left')

for j in range(len(CATEGORIES), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Test Period — 실제값 vs 예측값 (SMAPE)', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter: 예측값 vs 실제값 (모델별) ──────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (m, col) in zip(axes, col_map.items()):
    sub = pred_df.dropna(subset=[col])
    ax.scatter(sub['actual'], sub[col],
               alpha=0.25, s=6, color=MODEL_COLORS[m])

    # 이상적인 예측선 (y = x)
    lim = max(sub['actual'].max(), sub[col].max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', linewidth=0.8, label='perfect')

    overall = smape(sub['actual'], sub[col])
    ax.set_title(f'{m}  (전체 SMAPE = {overall:.2f}%)')
    ax.set_xlabel('실제값')
    ax.set_ylabel('예측값')
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.legend(fontsize=8)

plt.suptitle('Predicted vs Actual Scatter (전체 카테고리 합산)', y=1.02)
plt.tight_layout()
plt.show()

## 7. 오차 분포 — Boxplot & Violin

SMAPE 평균만 보면 이상치에 의한 왜곡을 놓칩니다.  
오차가 얼마나 고르게 분포하는지, 극단적으로 틀리는 날이 얼마나 있는지를 확인합니다.

In [ ]:
# 날짜별 pointwise SMAPE 계산
for m, col in col_map.items():
    sub = pred_df.dropna(subset=[col])
    pred_df.loc[sub.index, f'smape_pt_{m}'] = smape_pointwise(
        sub['actual'].values, sub[col].values
    )

smape_pt_cols = [f'smape_pt_{m}' for m in MODEL_KEYS]

# ── 전체 오차 분포 ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
ax = axes[0]
bp_data  = [pred_df[c].dropna().values for c in smape_pt_cols]
bp       = ax.boxplot(bp_data, patch_artist=True, notch=True,
                      medianprops=dict(color='black', linewidth=1.5))
for patch, m in zip(bp['boxes'], MODEL_KEYS):
    patch.set_facecolor(MODEL_COLORS[m])
    patch.set_alpha(0.7)
ax.set_xticklabels(MODEL_KEYS)
ax.set_ylabel('SMAPE (%)')
ax.set_title('오차 분포 Boxplot (전체 카테고리)')
ax.set_ylim(0, min(200, ax.get_ylim()[1]))

# 중앙값 표시
for i, m in enumerate(MODEL_KEYS):
    med = np.median(pred_df[f'smape_pt_{m}'].dropna())
    ax.text(i+1, med + 2, f'{med:.1f}%', ha='center', fontsize=8, fontweight='bold')

# Violin
ax = axes[1]
parts = ax.violinplot(bp_data, positions=range(1, len(MODEL_KEYS)+1),
                      showmedians=True, showextrema=False)
for pc, m in zip(parts['bodies'], MODEL_KEYS):
    pc.set_facecolor(MODEL_COLORS[m])
    pc.set_alpha(0.6)
ax.set_xticks(range(1, len(MODEL_KEYS)+1))
ax.set_xticklabels(MODEL_KEYS)
ax.set_ylabel('SMAPE (%)')
ax.set_title('오차 분포 Violin (전체 카테고리)')
ax.set_ylim(0, min(200, ax.get_ylim()[1]))

plt.tight_layout()
plt.show()

In [ ]:
# ── 카테고리별 오차 분포 Boxplot ─────────────────────
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
axes = axes.flatten()

for i, cat in enumerate(CATEGORIES):
    ax  = axes[i]
    sub = pred_df[pred_df['category'] == cat]

    bp_data = [sub[c].dropna().values for c in smape_pt_cols]
    bp      = ax.boxplot(bp_data, patch_artist=True, notch=False,
                         medianprops=dict(color='black', linewidth=1.5),
                         flierprops=dict(marker='.', markersize=3, alpha=0.4))
    for patch, m in zip(bp['boxes'], MODEL_KEYS):
        patch.set_facecolor(MODEL_COLORS[m])
        patch.set_alpha(0.7)

    ax.set_xticklabels(MODEL_KEYS, fontsize=8)
    ax.set_ylabel('SMAPE (%)', fontsize=8)
    ax.set_title(cat, fontsize=9)
    ax.set_ylim(0, min(200, ax.get_ylim()[1]))

for j in range(len(CATEGORIES), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('카테고리별 오차 분포 Boxplot', y=1.01, fontsize=11)
plt.tight_layout()
plt.show()

## 8. 날짜별 오차 추이

특정 날짜(이벤트, 이상치)에서 세 모델의 오차가 어떻게 움직이는지 확인합니다.  
SMAPE 평균이 비슷해도 오차 발생 패턴이 다를 수 있습니다.

In [ ]:
# 전체 카테고리 합산 — 날짜별 평균 SMAPE 추이
daily_smape = (
    pred_df
    .groupby('date')[smape_pt_cols]
    .mean()
    .reset_index()
)
daily_smape.columns = ['date'] + MODEL_KEYS

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# 상단: 날짜별 SMAPE 추이 (7일 이동평균 포함)
ax = axes[0]
for m in MODEL_KEYS:
    ax.plot(daily_smape['date'], daily_smape[m],
            color=MODEL_COLORS[m], alpha=0.25, linewidth=0.7)
    # 7일 이동평균
    ax.plot(daily_smape['date'],
            daily_smape[m].rolling(7, center=True).mean(),
            color=MODEL_COLORS[m], linewidth=1.5, label=f'{m} (7일 이동평균)')

ax.set_ylabel('SMAPE (%)')
ax.set_title('날짜별 평균 SMAPE 추이 (전체 카테고리)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
ax.tick_params(axis='x', rotation=40)
ax.legend(fontsize=8)

# 하단: 모델간 SMAPE 격차 (XGBoost - ARIMA, LSTM - ARIMA)
ax = axes[1]
ax.plot(daily_smape['date'],
        daily_smape['XGBoost'] - daily_smape['ARIMA'],
        color=MODEL_COLORS['XGBoost'], linewidth=1,
        label='XGBoost - ARIMA')
ax.plot(daily_smape['date'],
        daily_smape['LSTM'] - daily_smape['ARIMA'],
        color=MODEL_COLORS['LSTM'], linewidth=1,
        label='LSTM - ARIMA')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.fill_between(daily_smape['date'],
                daily_smape['XGBoost'] - daily_smape['ARIMA'],
                0, alpha=0.1, color=MODEL_COLORS['XGBoost'])
ax.set_ylabel('SMAPE 격차 (%)')
ax.set_title('ARIMA 대비 오차 격차 (양수 = ARIMA가 더 좋음)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
ax.tick_params(axis='x', rotation=40)
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# 오차가 가장 컸던 날짜 Top 10
print("=" * 60)
print("모델별 — 오차 최대 날짜 Top 10")
print("=" * 60)

for m in MODEL_KEYS:
    top10 = (
        pred_df[['date', 'category', f'smape_pt_{m}']]
        .dropna()
        .sort_values(f'smape_pt_{m}', ascending=False)
        .head(10)
    )
    print(f"\n[ {m} ]")
    print(top10.to_string(index=False))

## 9. 종합 인사이트

분석 결과를 바탕으로 핵심 질문에 대한 답변을 준비합니다.

In [ ]:
print("=" * 60)
print("예측 성능 비교 요약")
print("=" * 60)

print("\n[ 1. 전체 평균 SMAPE ]")
for m in MODEL_KEYS:
    avg  = smape_df[f'smape_{m}'].mean()
    med  = pred_df[f'smape_pt_{m}'].dropna().median()
    std  = pred_df[f'smape_pt_{m}'].dropna().std()
    print(f"  {m:<10}: 평균={avg:.2f}%  중앙값={med:.2f}%  표준편차={std:.2f}%")

print("\n[ 2. 모델별 1위 카테고리 수 ]")
win_counts = smape_df['best_model'].value_counts()
for m in MODEL_KEYS:
    print(f"  {m:<10}: {win_counts.get(m, 0)}개")

print("\n[ 3. 카테고리 난이도 (SMAPE 평균 기준) ]")
diff_df = smape_df[['category', 'smape_mean']].sort_values('smape_mean', ascending=False)
print("  예측 어려운 카테고리:")
print(diff_df.head(3).to_string(index=False))
print("  예측 쉬운 카테고리:")
print(diff_df.tail(3).to_string(index=False))

print()
print("=" * 60)
print("다음 단계: 05_cost_simulation.ipynb")
print("핵심 질문 — SMAPE가 낮은 모델이 비용도 낮은가?")
print("=" * 60)